In [ ]:
import json
import csv
import math
import time
from pathlib import Path
from openai import OpenAI
from langchain_chroma import Chroma

In [ ]:
DOCS_DIR        = "../../data/parsed_json/"
GOLDEN_SET_PATH = "../../eval/golden_dataset/golden_dataset_v2.csv"
COLLECTION_NAME = "rfp_docs"
BATCH_SIZE      = 500
EMBEDDING_MODEL = "bge-m3"
CHUNK_SIZES     = [300, 500, 1000, 2000]
TOP_K           = 5

import os
os.environ["OPENAI_API_KEY"] = ""
client = OpenAI()

In [ ]:
GS_TO_DOCID = {
    "GKL_그룹웨어":            "D093",
    "KUSF_체육":               "D011",
    "강릉어선안전":             "D024",
    "경기_사회서비스":          "D087",
    "고려대학교_차세대포털":    "D008",
    "광주과기원_RCMS":          "D073",
    "광주과학기술원_학사시스템": "D039",
    "구미_육상":               "D018",
    "국립중앙의료원_응급":      "D069",
    "국민연금공단_이러닝":      "D049",
    "국민연금_멀티턴1":         "D050",
    "국민연금_멀티턴2":         "D050",
    "국민연금_멀티턴3":         "D050",
    "국방_대용량":             "D010",
    "기초과학연구원_극저온":    "D051",
    "나노종합_팹":             "D099",
    "대검찰청_홈페이지":        "D053",
    "민속박물관_아카이브":      "D090",
    "벤처협회_시스템":          "D086",
    "보험개발원_실손":          "D083",
    "봉화군_재난":             "D005",
    "부산관광_ERP":            "D037",
    "서민금융_채팅":            "D056",
    "서영대_교육":             "D045",
    "서울_디지털성범죄":        "D068",
    "서울_지도플랫폼":          "D040",
    "서울교육청_ISP":           "D084",
    "세종_인사":               "D088",
    "우즈벡_관개":             "D072",
    "울산_버스":               "D034",
    "인천_도시계획":            "D004",
    "인천_일자리":             "D030",
    "인천공항_ERP":            "D079",
    "적십자_재해복구":          "D095",
    "철도_ISP":               "D070",
    "통합정보시스템_충돌":      "D016",
    "평택_버스":               "D060",
    "해양박물관_자료":          "D066",
    "고려대_vs_광주과기원":     ["D008", "D039"],
    "버스_다중비교":            ["D034", "D060"],
    "재난안전_종합":            ["D005", "D007"],
    "철도_vs_인천_ISP":        ["D070", "D030"],
    "TEST": None, "unknown": None, "ISP_다중비교": None,
    "교육관련_다중문서": None, "문화_다중비교": None,
    "의료_다중문서": None, "재공고_종합": None,
    "신규_vs_고도화": None, "예산_최소_최대": None,
    "멀티턴_심화1": None, "멀티턴_심화2": None,
    "모른다_테스트1": None, "모른다_테스트2": None,
    "모른다_테스트3": None, "모른다_테스트4": None,
    "모른다_테스트5": None, "모른다_테스트6": None,
    "존재하지않는사업": None, "입찰마감_확인": None,
}


1. 유틸리티

In [ ]:
def clean(val):
    # NaN / None → 빈 문자열 (Chroma 메타데이터는 NaN·None 불가)
    if val is None:
        return ""
    if isinstance(val, float) and math.isnan(val):
        return ""
    return val

In [ ]:
def build_payload(doc: dict, section: dict, block: dict) -> dict:
    # 청크 메타데이터(페이로드) 생성
    meta = doc.get("metadata", {})
    return {
        "doc_id":        str(clean(doc.get("doc_id"))),
        "file_name":     str(clean(doc.get("file_name"))),
        "source_format": str(clean(doc.get("source_format"))),
        "사업명":         str(clean(meta.get("사업명"))),
        "발주기관":       str(clean(meta.get("발주기관"))),
        "사업유형":       str(clean(meta.get("사업유형"))),
        "사업금액":       float(clean(meta.get("사업금액")) or 0.0),
        "공고번호":       str(clean(meta.get("공고번호"))),
        "공고차수":       float(clean(meta.get("공고차수")) or 0.0),
        "공개일자":       str(clean(meta.get("공개일자"))),
        "입찰참여시작일":  str(clean(meta.get("입찰참여시작일"))),
        "입찰참여마감일":  str(clean(meta.get("입찰참여마감일"))),
        "재공고여부":     bool(meta.get("재공고여부", False)),
        "linked_doc_id": str(clean(meta.get("linked_doc_id"))),
        "사업요약":       str(clean(meta.get("사업요약"))),
        "header_path":   " > ".join(section.get("header_path", [])),
        "section_id":    str(clean(section.get("section_id"))),
        "block_id":      str(clean(block.get("block_id"))),
        "block_type":    str(clean(block.get("type"))),
        "table_type":    str(clean(block.get("table_type"))),
    }

2. 청킹

In [ ]:
def chunk_section(section: dict) -> list[dict]:

    header_prefix = " > ".join(section.get("header_path", []))
    results = []
    current_text = ""
    last_text_block = None

    for block in section.get("blocks", []):
        if block.get("is_decorative"):
            continue
        if block["type"] == "table":
            # 쌓인 텍스트 먼저 방출
            if current_text.strip():
                results.append({
                    "content": f"[섹션: {header_prefix}]\n\n{current_text.strip()}",
                    "block":   last_text_block,
                })
                current_text = ""
                last_text_block = None
            # 표 단독 방출
            results.append({
                "content": f"[섹션: {header_prefix}]\n\n{block['content']}",
                "block":   block,
            })
        else:
            # 텍스트 블록 누적
            if len(current_text) + len(block["content"]) > MAX_CHUNK_SIZE and current_text.strip():
                results.append({
                    "content": f"[섹션: {header_prefix}]\n\n{current_text.strip()}",
                    "block":   last_text_block,
                })
                current_text = block["content"] + "\n\n"
            else:
                current_text += block["content"] + "\n\n"
            last_text_block = block

    # 남은 텍스트 방출
    if current_text.strip():
        results.append({
            "content": f"[섹션: {header_prefix}]\n\n{current_text.strip()}",
            "block":   last_text_block,
        })

    return results

In [ ]:
def process_doc(doc: dict) -> tuple[list[str], list[dict]]:
    # JSON 문서 1개 → (청크 텍스트 리스트, 메타데이터 리스트)
    texts, metas = [], []

    # extraction_warnings 있으면 경고 출력 후 계속 처리
    warnings = doc.get("qa", {}).get("extraction_warnings", [])
    if warnings:
        print(f"  [WARN] {doc['doc_id']} — extraction_warnings: {warnings}")

    meta = doc.get("metadata", {})
    summary = str(clean(meta.get("사업요약")))
    사업명 = str(clean(meta.get("사업명")))
    발주기관 = str(clean(meta.get("발주기관")))

    for section in doc.get("sections", []):
        chunks = chunk_section(section)
        for item in chunks:
            # 사업명·발주기관·요약을 앞에 붙여 retrieval 정확도 향상
            prefix = (
                f"[사업명] {사업명}\n"
                f"[발주기관] {발주기관}\n"
                f"[요약] {summary}\n\n"
            )
            texts.append(prefix + item["content"])
            metas.append(build_payload(doc, section, item["block"] or {}))
    return texts, metas

3. 임베딩 모델

In [ ]:
def load_embedding_model(name: str):
    if name == "text-embedding-3-small":
        return OpenAIEmbeddings(model="text-embedding-3-small")
    elif name == "text-embedding-3-large":
        return OpenAIEmbeddings(model="text-embedding-3-large")
    elif name == "bge-m3":
        from langchain_huggingface import HuggingFaceEmbeddings
        return HuggingFaceEmbeddings(
            model_name="BAAI/bge-m3",
            model_kwargs={"device": "cuda"},
            encode_kwargs={"normalize_embeddings": True},
        )
    elif name == "ko-sroberta-multitask":
        from langchain_huggingface import HuggingFaceEmbeddings
        return HuggingFaceEmbeddings(
            model_name="snunlp/KR-SBERT-V40K-klueNLI-augSTS",
            model_kwargs={"device": "cuda"},
            encode_kwargs={"normalize_embeddings": True},
        )
    elif name == "embed-multilingual-v3":
        from langchain_cohere import CohereEmbeddings
        return CohereEmbeddings(model="embed-multilingual-v3.0")
    else:
        raise ValueError(f"알 수 없는 임베딩 모델: {name}")


4. Vector DB / Retrieval

5. 실행

6. 검색 테스트

In [ ]:
def llm_judge(question: str, answer: str, contexts: list[str]) -> dict:
    context_text = "\n\n---\n\n".join(contexts)
    prompt = f"""아래 질문, 정답, 검색된 컨텍스트를 보고 두 항목을 평가하세요.

질문: {question}
정답: {answer}
검색된 컨텍스트:
{context_text}

평가 기준:
- context_recall: 정답 내용이 검색된 컨텍스트에 얼마나 포함되어 있는가 (0.0~1.0)
- context_precision: 검색된 컨텍스트 중 질문에 실제로 관련 있는 비율 (0.0~1.0)

반드시 아래 JSON 형식으로만 답하세요:
{{"context_recall": 0.0, "context_precision": 0.0}}"""

    wait = 10
    for attempt in range(6):
        try:
            time.sleep(0.5)
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
            )
            return json.loads(response.choices[0].message.content)
        except Exception as e:
            if "RateLimitError" in type(e).__name__ or "rate_limit" in str(e).lower():
                print(f"  [RateLimit] {wait}초 대기 후 재시도 ({attempt+1}/6)")
                time.sleep(wait)
                wait = min(wait * 2, 120)
            else:
                break
    return {"context_recall": 0.0, "context_precision": 0.0}

7. 임베딩 모델 비교

8. 청크 크기 실험

In [ ]:
golden_set = []
with open(GOLDEN_SET_PATH, encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        golden_set.append(row)

golden_set = golden_set[:101]
print(f"골든셋 문항 수: {len(golden_set)}")

In [ ]:
embedding_model = load_embedding_model(EMBEDDING_MODEL)
eval_summary = []

for size in CHUNK_SIZES:
    chroma_dir = f"../../data/chroma_db_{size}"

    if not Path(chroma_dir).exists():
        print(f"[SKIP] chroma_db_{size} — DB 없음")
        continue

    print(f"\n[EVAL] chunk_size={size}")
    vs = Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=embedding_model,
        persist_directory=chroma_dir,
    )

    acc, recall, precision = [], [], []
    skipped = 0

    for row in golden_set:
        gs_key = str(row["doc_id"])
        target = GS_TO_DOCID.get(gs_key)

        if target is None:
            skipped += 1
            continue

        target_ids = target if isinstance(target, list) else [target]

        hits = vs.similarity_search_with_relevance_scores(row["question"], k=TOP_K)
        retrieved_doc_ids = [doc.metadata.get("doc_id", "") for doc, _ in hits]
        contexts = [doc.page_content for doc, _ in hits]

        acc.append(1.0 if hit_any(retrieved_doc_ids, target_ids) else 0.0)
        scores = llm_judge(row["question"], row["answer"], contexts)
        recall.append(scores["context_recall"])
        precision.append(scores["context_precision"])

    n = len(acc)
    print(f"  완료 | 평가: {n}개 | 제외: {skipped}개")

    eval_summary.append({
        "청크 크기":          size,
        "retrieval_accuracy": round(sum(acc)       / n, 4),
        "context_recall":     round(sum(recall)    / n, 4),
        "context_precision":  round(sum(precision) / n, 4),
    })


In [ ]:
from IPython.display import Markdown, display

rows = "\n".join(
    f"| {r['청크 크기']} | {r['retrieval_accuracy']} | {r['context_recall']} | {r['context_precision']} |"
    for r in eval_summary
)
display(Markdown(f"""## 골든셋 평가 결과

| 청크 크기 | retrieval_accuracy | context_recall | context_precision |
|----------|-------------------|----------------|-------------------|
{rows}
"""))